In [ ]:
!pip install plotly pandas numpy -q

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'
print("Библиотеки загружены!")

Библиотеки загружены!


In [ ]:
# Генерация банковского датасета (как в предыдущих уроках)
np.random.seed(42)
N = 2000
cities = ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург']
df = pd.DataFrame({
    'client_id': range(1001, 1001 + N),
    'city': np.random.choice(cities, N),
    'age': np.random.randint(21, 70, N),
    'income': np.random.normal(50000, 20000, N).astype(int),
    'loan_amount': np.random.normal(200000, 80000, N).astype(int),
    'credit_score': np.random.normal(650, 100, N).astype(int),
    'default': np.random.choice([0, 1], N, p=[0.9, 0.1])
})
# Небольшая очистка
df['income'] = df['income'].clip(lower=0)
df['loan_amount'] = df['loan_amount'].clip(lower=0)
print(f"Датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
df.head()

Датасет: 2000 строк, 7 колонок


,client_id,city,age,income,loan_amount,credit_score,default
0,1001,Новосибирск,48,20976,216762,913,0
1,1002,Екатеринбург,27,49758,112669,680,0
2,1003,Казань,52,24952,218707,611,0
3,1004,Екатеринбург,31,57272,282835,429,0
4,1005,Екатеринбург,30,67737,133115,788,0


## 🔍 Scatter plot — доход vs сумма кредита

Каждая точка — клиент. Цвет точки показывает возраст, размер — кредитный скоринг.
При наведении курсора всплывает ID и город. График можно приближать и двигать.

In [ ]:
#   px.scatter() — интерактивный scatter plot
#   x='income', y='loan_amount' — оси
#   color='age' — цвет точек зависит от возраста
#  size='credit_score' — размер точки по скорингу
#   hover_name='client_id' — при наведении покажем ID
#   hover_data=['city'] — дополнительно покажем город
#   title — заголовок графика
#   labels — переименование подписей осей

fig = px.scatter(df,
                 x='income', y='loan_amount',
                 color='age',
                 size='credit_score',
                 hover_name='client_id',
                 hover_data=['city'],
                 title='Клиенты: доход vs сумма кредита',
                 labels={'income':'Доход, руб.', 'loan_amount':'Сумма кредита, руб.'})
fig.show()

**ИНТЕРПРЕТАЦИЯ:**
- Наведи мышь на точку — увидишь подсказку с полной информацией.
- Выдели область — график увеличится.
- Самые светлые точки — старшие клиенты, самые тёмные — молодые.

## 📈 Line plot — динамика выдач кредитов
# Создадим временной ряд: 30 дней ежедневных выдач и нарисуем линию с точками.

In [ ]:
#   Создаём временной ряд: даты и случайные суммы
#   px.line() — интерактивный линейный график
#║  markers=True — добавить точки на линии

dates = pd.date_range('2026-01-01', periods=30, freq='D')
daily = pd.DataFrame({
    'date': dates,
    'amount': np.random.normal(200000, 30000, 30).astype(int),
    'type': 'fact'  # фактическое значение
})

fig = px.line(daily, x='date', y='amount', title='Ежедневные выдачи кредитов (Plotly)',
              markers=True, labels={'amount':'Сумма, руб.'})
fig.show()

**ИНТЕРПРЕТАЦИЯ:**
- Наведи на точку — увидишь точную сумму и дату.
- Можно добавить вторую линию (план) и сравнить.

## Bar chart — средний доход по городам и типу клиента
Строим сгруппированные столбцы. Plotly автоматически добавит подсказки.

In [ ]:
#   Сначала группируем, потом px.bar()
#   barmode='group' — группированные столбцы
#   color='default' — разные цвета для плательщиков и
#   дефолтников

grouped = df.groupby(['city', 'default'])['income'].mean().reset_index()
grouped['default'] = grouped['default'].map({0:'Плательщик', 1:'Дефолтник'})

fig = px.bar(grouped, x='city', y='income', color='default',
             barmode='group', title='Средний доход по городам и типу клиента',
             labels={'income':'Средний доход, руб.'})
fig.show()

**ИНТЕРПРЕТАЦИЯ:**
- Наведи на столбец — увидишь точное среднее, город и тип клиента.
- Можно кликнуть в легенде, чтобы оставить только дефолтников или только плательщиков.

## Histogram и Box plot — распределения с интерактивом
Гистограмма доходов и boxplot по городам с цветовым выделением дефолта.

In [ ]:
# Гистограмма
fig = px.histogram(df, x='income', nbins=30, title='Интерактивная гистограмма доходов',
                   labels={'income':'Доход, руб.'})
fig.show()

In [ ]:
# Boxplot дохода по городам с разделением по default
fig = px.box(df, x='city', y='income', color='default',
             title='Доход по городам и типу клиента (Plotly Box)',
             labels={'income':'Доход, руб.'})
fig.show()

## Scatter 3D — трёхмерная сцена данных
3 переменные: доход, сумма кредита, возраст. Цвет — дефолт.

In [ ]:
#   px.scatter_3d() — трёхмерный scatter plot
#   x='income', y='loan_amount', z='age' — три оси
#   color='default' — цвет по наличию дефолта
#   hover_name='client_id' — подсказка с ID

fig = px.scatter_3d(df,
                    x='income', y='loan_amount', z='age',
                    color='default',
                    hover_name='client_id',
                    title='3D-визуализация клиентов (Plotly)',
                    labels={'income':'Доход, руб.', 'loan_amount':'Сумма кредита, руб.', 'age':'Возраст'})
fig.show()

**ИНТЕРПРЕТАЦИЯ:**
- Вращай сцену, зажав левую кнопку мыши.
- Приближай колёсиком.
- Синие точки — плательщики, красные — дефолтники.
- Ищи кластеры красных точек — например, низкий доход, высокий кредит, молодой возраст.

## 🎸 ТВОРЧЕСКОЕ ЗАДАНИЕ

1. Построй scatter plot с color='city' и size='age'.
2. Создай line plot для столбца 'credit_score' (добавь второй столбец со скользящим средним).
3. Построй bar chart количества клиентов по городам (value_counts) — интерактивные столбцы.
4. Сделай box plot суммы кредита по возрастным группам (как в уроке 4.2, но через Plotly).
5. Сгенерируй 3D-график с color='city', включи hover_data=['default'].

In [ ]:
# пиши свой код